# OEM Fleet Recall Propagation - Demo Notebook

Seven-act walkthrough of the recall-propagation demo on top of `CARS_DEMO.FLEET` (325 vehicles, 762 service events, 254 recall assignments, 5 active campaigns, 27 centre-to-centre handoffs). Each act uses a different RAI reasoner family on the same `cars` PyRel ontology.

| Act | Reasoner      | Question                                                            |
|-----|---------------|----------------------------------------------------------------------|
| 1   | Rules          | Recall SLA compliance audit (severity Code 1 / 2 past completion)    |
| 2   | Graph          | Defect cascade from Continental MK C1 brake booster (supplier -> VIN) |
| 3   | Heuristic      | Per-VIN urgency ranking, top 20                                      |
| 4   | Prescriptive   | Assign open recall jobs to centres for next 4 weeks (HiGHS MIP)      |
| 5   | Persistent     | Operator adds 'prioritise prior-accident VINs' rule, MIP re-solves   |
| 6   | Pathfinder     | Multi-hop centre-to-centre referral chains (3-way handoff data)      |
| 7   | Graph + MIP    | Louvain cohorts feed a per-community late-share cap into a new MIP   |

Runtime estimate: 8-10 minutes warm. The first cell triggers engine resume if `cars_logic_l` is suspended (+3-5 min).


In [1]:
# Add the repo root to sys.path. Walks up from CWD looking for the
# rai_code/ directory so the notebook works whether VSCode opens it
# with CWD = repo root, CWD = rai_code/manual/, or anywhere in between.
import sys, os
from pathlib import Path
_cwd = Path(os.getcwd()).resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / 'rai_code' / 'manual' / 'cars.py').exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        os.chdir(_p)
        print(f'repo root: {_p}')
        break
else:
    raise RuntimeError(f'could not locate repo root from {_cwd}')

# Disarm PyRel's running-loop guard so synchronous Problem.solve() works
# inside Jupyter (mirrors the supply_chain_demo pattern).
import relationalai.client as _ra_client
import relationalai.services.reasoners.client as _ra_reasoners_client
_noop = lambda *a, **k: None
_ra_client.raise_if_running_event_loop = _noop
_ra_reasoners_client.raise_if_running_event_loop = _noop

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

# Importing the ontology triggers all model.define() statements.
from rai_code.manual.cars import (
    Supplier, Part, BomNode, Vehicle, Owner, ServiceCentre, Region,
    RecallCampaign, RecallAssignment, OpenRecall, SLABreachedRecall,
    PriorAccident, PriorityVehicle, in_bom, model,
)
from rai_code.manual.demo_queries import (
    q1_recall_sla_audit, q1b_breached_by_centre,
    q2_continental_cascade, q2_regional_rollup, q2_top_centres,
    q3_urgency_top20,
    q4_assign_recall_jobs, q5_assign_recall_jobs_priority,
)
print('imports ok')

imports ok


## Act 1 - Rules: recall SLA compliance audit

> **The question (campaign manager types):** 'Show me every Open recall on a VIN whose owner-notification age has breached the campaign's completion window. Break it out by campaign and by responsible centre.'

The `SLABreachedRecall` derived concept encodes NHTSA 49 CFR 577 / KBA Rueckruf semantics once at the ontology layer: `OpenRecall AND age_days_at_demo > campaign.completion_days AND severity_code <= 2`. The query just counts.

In [2]:
df1 = q1_recall_sla_audit()
df1

,campaign,campaign_name,severity,breached_open
0,IBS-2024-A,Continental MK C1 brake-booster firmware v2.3.1,2,8
1,HVB-2024-A,Samsung SDI HV battery module thermal risk,1,7
2,EGR-2023-B,BorgWarner EGR cooler crack risk,2,3
3,AIRBAG-2022-A,Joyson PSAN inflator carry-over recall,1,1


In [3]:
fig = px.bar(
    df1, x='campaign', y='breached_open',
    color='severity',
    color_continuous_scale=[(0, '#c62828'), (0.5, '#ef6c00'), (1, '#f9a825')],
    range_color=(1, 3),
    title='SLA-breached Open recalls by campaign (severity 1 = stop driving, 2 = repair urgently)',
    text='breached_open',
    hover_data=['campaign_name'],
)
fig.update_traces(textposition='outside')
fig.update_layout(height=420, xaxis_title='Campaign', yaxis_title='SLA-breached Open recalls')
fig

In [4]:
df1b = q1b_breached_by_centre()
fig = px.bar(
    df1b, x='centre', y='breached_open', color='country',
    title='SLA-breached Open recalls by responsible centre',
    text='breached_open',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=460, xaxis_title='Service centre', yaxis_title='SLA-breached Open recalls', xaxis_tickangle=-30)
fig

**What to look at.** The Continental MK C1 brake-booster firmware campaign (IBS-2024-A) dominates the SLA-breach population. Severity-Code-1 campaigns (Samsung SDI HV battery) breach a smaller absolute number but matter more per VIN. Munich / Cologne / Hamburg lead by centre exposure: high-volume EU centres carry the largest open populations on the IBS firmware reflash work. None of this required the data team; the rule lives in the ontology as `SLABreachedRecall`.

## Act 2 - Graph: defect cascade from Continental MK C1

> **The question:** 'Continental notified us this morning of an expanded scope on the MK C1 brake-booster firmware defect. Walk the cascade: supplier -> part -> bom node -> VIN -> owner -> service centre. Roll up by region. Who do we have to notify, and where do they go?'

RAI's relational ontology walks the cascade as a single declarative join. The same query in SQL would be a 6-table CTE; the same in the source BMW AIR / iLEAD systems would be a multi-system handoff.

In [5]:
df2 = q2_continental_cascade()
print(f'affected VINs: {len(df2)}')
df2.head(15)

affected VINs: 67


,vin,model,plant,production_date,owner_country,nearest_centre
0,WBA00000000000001,iX1 xDrive30 (U11 LCI),Regensburg,2023-01-01,DE,BMW Cologne Service
1,WBA00000000000014,X1 M35i xDrive (U11),Regensburg,2024-01-01,BE,BMW Hamburg Service
2,WBA00000000000015,iX1 xDrive30 (U11 LCI),Regensburg,2023-01-01,CH,BMW Munich Service
3,WBA00000000000017,X1 M35i xDrive (U11),Regensburg,2024-03-02,DE,BMW Leipzig Service
4,WBA00000000000022,iX1 xDrive30 (U11 LCI),Regensburg,2023-01-01,AT,BMW Frankfurt Service
5,WBA00000000000026,X1 M35i xDrive (U11),Regensburg,2022-01-01,DE,BMW Cologne Service
6,WBA00000000000027,iX1 xDrive30 (U11 LCI),Regensburg,2023-01-01,DE,BMW Berlin Service
7,WBA00000000000029,3 Series 330i (G21 LCI),Munich,2024-03-20,DE,BMW Hamburg Service
8,WBA00000000000050,X1 M35i xDrive (U11),Regensburg,2023-12-07,CH,BMW Hamburg Service
9,WBA00000000000051,iX1 xDrive30 (U11 LCI),Regensburg,2023-01-01,CH,BMW Munich Service


In [6]:
df2_rollup = q2_regional_rollup()
df2_centres = q2_top_centres()
fig = make_subplots(rows=1, cols=2, subplot_titles=('Regional rollup', 'Top centres by affected VINs'))
fig.add_trace(go.Bar(x=df2_rollup['rollup'], y=df2_rollup['affected_vins'], text=df2_rollup['affected_vins'], textposition='outside', name='affected VINs'), row=1, col=1)
fig.add_trace(go.Bar(x=df2_centres['centre'], y=df2_centres['affected_vins'], text=df2_centres['affected_vins'], textposition='outside', name='per centre', marker_color='#e57373'), row=1, col=2)
fig.update_xaxes(tickangle=-30, row=1, col=2)
fig.update_layout(height=460, showlegend=False, title_text='Continental MK C1 brake-booster cascade')
fig

In [7]:
# Sankey-style cascade: supplier -> part -> plant -> centre.
edges = []
for _, row in df2.iterrows():
    edges.append(('Continental AG', 'MK C1 booster ECU v2.3.x'))
    edges.append(('MK C1 booster ECU v2.3.x', f'Plant: {row["plant"]}'))
    edges.append((f'Plant: {row["plant"]}', f'Centre: {row["nearest_centre"]}'))
edge_counts = pd.Series(edges).value_counts()
nodes = sorted(set([s for s, _ in edge_counts.index] + [t for _, t in edge_counts.index]))
node_idx = {n: i for i, n in enumerate(nodes)}
fig = go.Figure(data=[go.Sankey(
    node=dict(label=nodes, pad=14, thickness=18, color='#5e81ac'),
    link=dict(
        source=[node_idx[s] for s, _ in edge_counts.index],
        target=[node_idx[t] for _, t in edge_counts.index],
        value=edge_counts.values,
    ),
)])
fig.update_layout(height=520, title_text=f'Cascade: Continental -> MK C1 ECU -> Plants -> Service Centres ({len(df2)} VINs)')
fig

**What to look at.** From a single supplier-part input, the agent enumerates every affected VIN, every owner country, and every service centre that has to absorb the work. The regional rollup (EU / NA / LATAM) is the after-sales head's slide; the centre-level breakdown is the dealer-network operations manager's. Both come from the same query, because both views are projections of the same ontology join.

## Act 2b - Graph view: cohort communities (Louvain)

The cascade in Act 2 is a relational join. What does the *graph* of vehicle exposure look like? Two VINs share an edge when they share a BOM node. Louvain community detection (run on `cars_logic_l`) partitions the population into natural cohorts - groups of vehicles that move together when a supplier issue hits.

Different from Act 2: this is the Graph reasoner, not a relational join. Different from Act 3: this is structural (who clusters with whom), not score-based ranking.

In [8]:
from rai_code.manual.demo_queries import (
    q7_vehicle_communities, q7_vehicle_communities_nodes_and_edges,
)

comm_summary = q7_vehicle_communities()
print(f'communities: {len(comm_summary)}    largest: {int(comm_summary.vins.max())} VINs    smallest: {int(comm_summary.vins.min())} VINs')
comm_summary.head(10)

✅ Done

Parallel init finished in            1ms
────────────────────────────────────────

communities: 139    largest: 75 VINs    smallest: 1 VINs


,community,vins,dominant_model
0,1,75,i7 M70 xDrive
1,3,70,X1 M35i xDrive (U11)
2,2,28,iX M60
3,4,17,3 Series 330i (G21 LCI)
4,97,1,X7 xDrive40d (G07 LCI)
5,91,1,XM Label Red
6,92,1,i4 M50 (G26 LCI)
7,93,1,M2 (G87 LCI)
8,94,1,XM Label Red
9,95,1,i4 M50 (G26 LCI)


In [9]:
import networkx as nx

nodes, edges = q7_vehicle_communities_nodes_and_edges()

# Build networkx graph for force-directed layout. Spring layout
# with seed=42 keeps the picture reproducible across runs.
G = nx.Graph()
for _, r in nodes.iterrows():
    G.add_node(r['vin'])
for _, r in edges.iterrows():
    G.add_edge(r['v1'], r['v2'], weight=int(r['shared_boms']))
pos = nx.spring_layout(G, seed=42, k=0.55, iterations=80, weight='weight')

# Edge trace: single Scatter with None-separated segments. Edge
# width and alpha scale with the count of shared BOM nodes so the
# tightest cohort links visually pop.
edge_x, edge_y, edge_alpha, edge_w = [], [], [], []
max_sb = max(edges['shared_boms']) if len(edges) else 1
for _, r in edges.iterrows():
    x0, y0 = pos[r['v1']]
    x1, y1 = pos[r['v2']]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.6, color='rgba(140,140,140,0.25)'),
    hoverinfo='skip', showlegend=False,
)

# Node trace - colour by community, size by degree in the
# BOM-sharing graph, hover-text with every meaningful attribute.
node_x = [pos[v][0] for v in nodes['vin']]
node_y = [pos[v][1] for v in nodes['vin']]
node_color = nodes['community'].tolist()
node_size = (nodes['degree'].clip(lower=1, upper=40) * 0.45 + 9).tolist()
hovertext = [
    f'<b>{r.vin}</b><br>'
    f'{r.model}<br>'
    f'plant: {r.plant} | fuel: {r.fuel}<br>'
    f'mileage: {int(r.mileage):,} km<br>'
    f'community: {r.community} | degree: {int(r.degree)}<br>'
    f'campaigns: {int(r.campaign_count)} | open recalls: {int(r.open_count)}<br>'
    f'accident: {r.accident}'
    for r in nodes.itertuples()
]
node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers',
    marker=dict(
        size=node_size,
        color=node_color,
        colorscale='Turbo',
        showscale=True,
        colorbar=dict(title=dict(text='Community', side='right'), x=1.02, len=0.8),
        line=dict(width=0.6, color='white'),
        opacity=0.92,
    ),
    text=hovertext, hoverinfo='text', showlegend=False,
)

n_communities = int(nodes['community'].nunique())
biggest = comm_summary.iloc[0]
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title=(
        f'Vehicle cohort communities (Louvain on shared-BOM graph)<br>'
        f"<sup>{len(nodes)} VINs - {len(edges):,} edges - {n_communities} communities. "
        f'Largest cluster: {int(biggest.vins)} VINs (dominant model: {biggest.dominant_model}). '
        f'Node colour = community; size = degree in BOM-sharing graph.</sup>'
    ),
    template='plotly_dark',
    plot_bgcolor='#0f1115', paper_bgcolor='#0f1115',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='y'),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    width=1100, height=760,
    margin=dict(l=10, r=10, t=90, b=10),
)
fig

▰▰▰▰ Submitting job...                            4.2s

✅ Done

Parallel init finished in           2.1s
 ➜ Provisioning                     2.1s
────────────────────────────────────────

✅ Done

Parallel init finished in            1ms
────────────────────────────────────────

**What to look at.** Each colour is a Louvain-detected cohort. The largest community usually clusters Munich + Regensburg + Spartanburg X-series and 3 Series LCI units that all share the Continental IBS BOM nodes - that is the structural Act 2 cascade visualised. Smaller clusters around the periphery are Dingolfing iX / i7 EVs (the HVB campaign cohort) and the lone diesel X1 (EGR cohort). Hover any node for VIN-level detail; tightly bound clusters have many BOM nodes in common and would move as a unit under a supplier expansion.

Why a graph view matters for the talk track: the cascade in Act 2 returned a *list*. This view shows the *structure*. The audience sees community boundaries that no SQL query exposes, and the operations team can identify VIN clusters with multi-supplier exposure at a glance.

## Act 3 - Heuristic: per-VIN urgency ranking

> **The question:** 'Of the ~120 Open recall jobs across all active campaigns, give me the top 20 by urgency. Weights: mileage as a utilisation proxy, owner-notification age as a calendar-time proxy, prior accident severity as a safety proxy, distance to the nearest equipped centre as a logistics proxy.'

The urgency formula lives on the ontology as a derived `RecallAssignment.urgency` Property. Deterministic, auditable, defensible to an insurance reviewer in a way a black-box GNN is not. Per the talk-track disclaimer this stands in for the GNN-based Predictive reasoner which is still preview.

In [10]:
df3 = q3_urgency_top20()
df3

,recall_id,vin,model,plant,campaign,mileage,age_days,distance_km,accident,urgency
0,REC-000001,WBA00000000000001,iX1 xDrive30 (U11 LCI),Regensburg,IBS-2024-A,214084,176,189,Collision,0.730575
1,REC-000013,WBA00000000000068,X1 M35i xDrive (U11),Regensburg,IBS-2024-A,105013,182,187,Collision,0.600544
2,REC-000212,WBA00000000000068,X1 M35i xDrive (U11),Regensburg,STARTER-2024-A,105013,74,187,Collision,0.563558
3,REC-000058,WBA00000000000256,iX1 xDrive30 (U11 LCI),Regensburg,IBS-2024-A,95230,167,80,Rear-end,0.519468
4,REC-000091,WBA00000000000164,i7 M70 xDrive,Dingolfing,HVB-2024-A,70475,59,143,Rear-end,0.490575
5,REC-000247,WBA00000000000256,iX1 xDrive30 (U11 LCI),Regensburg,STARTER-2024-A,95230,75,80,Rear-end,0.487961
6,REC-000011,WBA00000000000061,iX1 xDrive30 (U11 LCI),Regensburg,IBS-2024-A,66363,188,73,Rear-end,0.487819
7,REC-000173,WBA00000000000220,iX1 xDrive30 (U11 LCI),Regensburg,EGR-2023-B,208548,202,206,NaN,0.443036
8,REC-000044,WBA00000000000202,X1 M35i xDrive (U11),Regensburg,IBS-2024-A,9932,167,100,Collision,0.429110
9,REC-000211,WBA00000000000061,iX1 xDrive30 (U11 LCI),Regensburg,STARTER-2024-A,66363,7,73,Rear-end,0.425833


In [11]:
df3_plot = df3.copy()
df3_plot['label'] = df3_plot['vin'].str[-6:] + ' | ' + df3_plot['model'].str.slice(0, 22) + ' | ' + df3_plot['campaign']
fig = px.bar(
    df3_plot.sort_values('urgency'), y='label', x='urgency', orientation='h',
    color='accident', color_discrete_sequence=px.colors.qualitative.Set2,
    title='Top-20 Open recalls by urgency score',
    text='urgency', hover_data=['plant', 'mileage', 'age_days', 'distance_km'],
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(height=600, yaxis_title='Vehicle (VIN tail | model | campaign)', xaxis_title='Urgency score (0 = lowest)')
fig

**What to look at.** The top of the queue mixes Regensburg-built X1 M35i units with high mileage and prior accidents (the IBS booster firmware on top of a structurally-damaged chassis is the wrong combination to leave on the road) with Dingolfing iX / i7 EVs from the HVB-2024-A campaign. Notice the score is a defensible weighted sum, not a black box; every input is a PyRel-derived property and every weight is visible in `demo_queries.py`.

## Act 4 - Prescriptive: assign recall jobs to centres x 4 weeks

> **The question:** 'Schedule the next four weeks of recall work. Each open job goes to exactly one centre in exactly one week. Minimise total urgency-weighted lateness vs. week 1. Constraints: per-centre weekly technician-hour capacity, per-centre per-week parts-stock for each campaign, no VIN assigned to a centre without the required tooling certification. Solve it.'

HiGHS MIP. Decision: x[recall, centre, week] in {0,1}. Constraints are stitched together from the ontology - they reference `tech_hours_available`, `on_hand_units`, `hv_certified`, `ibs_certified`, `body_shop` directly. Tooling-certification eligibility is pre-filtered into `JobAssignment` rows so the LP enumerates a feasible grid only.

In [12]:
df4, si4 = q4_assign_recall_jobs()
print(f'status: {si4.termination_status}    objective: {si4.objective_value:.2f}    solve time: {si4.solve_time_sec:.2f}s    jobs scheduled: {len(df4)}')
df4.head(15)

status: OPTIMAL    objective: 0.67    solve time: 0.14s    jobs scheduled: 121


,recall_id,vin,plant,campaign,centre,week,urgency,hours
0,REC-000001,WBA00000000000001,Regensburg,IBS-2024-A,BMW Cologne Service,1,0.730575,2.5
1,REC-000013,WBA00000000000068,Regensburg,IBS-2024-A,BMW Cologne Service,1,0.600544,2.5
2,REC-000212,WBA00000000000068,Regensburg,STARTER-2024-A,BMW Cologne Service,1,0.563558,3.0
3,REC-000058,WBA00000000000256,Regensburg,IBS-2024-A,BMW Regensburg Service,1,0.519468,2.5
4,REC-000091,WBA00000000000164,Dingolfing,HVB-2024-A,BMW Los Angeles Service,1,0.490575,8.0
5,REC-000247,WBA00000000000256,Regensburg,STARTER-2024-A,BMW Monterrey Service,1,0.487961,3.0
6,REC-000011,WBA00000000000061,Regensburg,IBS-2024-A,BMW Dallas Service,1,0.487819,2.5
7,REC-000173,WBA00000000000220,Regensburg,EGR-2023-B,BMW Munich Service,1,0.443036,5.0
8,REC-000044,WBA00000000000202,Regensburg,IBS-2024-A,BMW Miami Service,1,0.429110,2.5
9,REC-000211,WBA00000000000061,Regensburg,STARTER-2024-A,BMW Cologne Service,1,0.425833,3.0


In [13]:
# Stacked bar: jobs per (centre, week)
df4_pivot = df4.groupby(['centre', 'week']).size().reset_index(name='jobs')
fig = px.bar(
    df4_pivot, x='centre', y='jobs', color='week',
    color_continuous_scale='Viridis',
    title='Recall job assignments by centre and week (Act 4 baseline)',
    barmode='stack',
)
fig.update_layout(height=520, xaxis_tickangle=-30, yaxis_title='Jobs scheduled')
fig

In [14]:
# Week-by-week urgency distribution.
fig = px.box(
    df4, x='week', y='urgency', color='campaign',
    title='Urgency distribution by scheduled week (Act 4)',
    points='all',
)
fig.update_layout(height=460, xaxis_title='Week (1 = earliest)', yaxis_title='Urgency score')
fig

**What to look at.** The MIP pulls high-urgency jobs to week 1 and lets lower-urgency jobs slip later. Centres without the right tooling certification carry zero jobs from campaigns that need it (e.g. Monterrey is excluded from HV and IBS work). The objective is total urgency-weighted lateness; the binding constraint in practice is parts stock on the IBS campaign, not technician hours.

## Act 5 - Persistent rule: operator adds a safety priority

> **The question:** 'Add this rule to the ontology and re-solve: any Open recall on a VIN with a prior collision or rear-end accident is prioritised. Force the prioritised VIN to be scheduled no later than week 2 of the four-week horizon. Show me what changes.'

This is the institutional-knowledge moment. The rule is written once as the `PriorityVehicle` derived concept plus one extra `problem.satisfy(...)` constraint on the MIP. It now lives in the ontology - every downstream reasoner respects it without redeployment. The MIP re-solves with the new constraint and returns OPTIMAL.

In [15]:
df5, si5 = q5_assign_recall_jobs_priority()
print(f'status: {si5.termination_status}    objective: {si5.objective_value:.2f}    solve time: {si5.solve_time_sec:.2f}s    jobs scheduled: {len(df5)}')
df5.head(15)

status: OPTIMAL    objective: 0.67    solve time: 0.13s    jobs scheduled: 121


,recall_id,vin,plant,campaign,centre,week,urgency,hours
0,REC-000001,WBA00000000000001,Regensburg,IBS-2024-A,BMW Spartanburg Service,1,0.730575,2.5
1,REC-000013,WBA00000000000068,Regensburg,IBS-2024-A,BMW New York Service,1,0.600544,2.5
2,REC-000212,WBA00000000000068,Regensburg,STARTER-2024-A,BMW Cologne Service,1,0.563558,3.0
3,REC-000058,WBA00000000000256,Regensburg,IBS-2024-A,BMW Munich Service,1,0.519468,2.5
4,REC-000091,WBA00000000000164,Dingolfing,HVB-2024-A,BMW Frankfurt Service,1,0.490575,8.0
5,REC-000247,WBA00000000000256,Regensburg,STARTER-2024-A,BMW Monterrey Service,1,0.487961,3.0
6,REC-000011,WBA00000000000061,Regensburg,IBS-2024-A,BMW Dallas Service,1,0.487819,2.5
7,REC-000173,WBA00000000000220,Regensburg,EGR-2023-B,BMW Monterrey Service,1,0.443036,5.0
8,REC-000044,WBA00000000000202,Regensburg,IBS-2024-A,BMW Stuttgart Service,1,0.429110,2.5
9,REC-000211,WBA00000000000061,Regensburg,STARTER-2024-A,BMW Monterrey Service,1,0.425833,3.0


In [16]:
wk4 = df4.groupby('week').size().reindex([1, 2, 3, 4], fill_value=0).rename('Act 4 (baseline)').reset_index()
wk5 = df5.groupby('week').size().reindex([1, 2, 3, 4], fill_value=0).rename('Act 5 (priority rule)').reset_index()
fig = make_subplots(rows=1, cols=2, subplot_titles=('Jobs per week', 'Objective: weighted lateness'))
fig.add_trace(go.Bar(name='Act 4', x=wk4['week'], y=wk4['Act 4 (baseline)'], marker_color='#5e81ac'), row=1, col=1)
fig.add_trace(go.Bar(name='Act 5', x=wk5['week'], y=wk5['Act 5 (priority rule)'], marker_color='#bf616a'), row=1, col=1)
fig.add_trace(go.Bar(name='objective', x=['Act 4', 'Act 5'], y=[si4.objective_value, si5.objective_value], marker_color=['#5e81ac', '#bf616a']), row=1, col=2)
fig.update_xaxes(title_text='Week', row=1, col=1)
fig.update_xaxes(title_text='Solve', row=1, col=2)
fig.update_yaxes(title_text='Jobs scheduled', row=1, col=1)
fig.update_yaxes(title_text='Total weighted lateness', row=1, col=2)
fig.update_layout(height=460, barmode='group', title_text='Act 4 vs Act 5 - rule effect on schedule and objective')
fig

## Act 6 - Path traversal: centre-to-centre handoff chains

> **The question:** 'When a centre runs out of parts, where does the workload actually go - one hop downstream or three?'

The `CentreHandoff` table is a true 3-way relationship (`from_centre x to_centre x campaign`). The ontology exposes it through an N-arity adapter relationship `refers_for` so PyRel's Pathfinder can walk variable-length chains over it. The query below seeds at Monterrey (SC-MTY) and enumerates every chain of length 1..3 carrying campaign IBS-2024-A. The data contains a planted 3-hop chain SC-MTY -> SC-LAX -> SC-DAL -> SC-SPB.


In [ ]:
from rai_code.manual.demo_queries import q11_handoff_chains, q11_handoff_chain_summary

df11 = q11_handoff_chains(seed_centre_id='SC-MTY', max_hops=3, campaign_filter='IBS-2024-A')
print(f'rows (one per (path, hop)): {len(df11)}')
df11.head(20)


In [ ]:
# Chain summary: one row per path, with the centres concatenated by length.
df11_sum = q11_handoff_chain_summary(seed_centre_id='SC-MTY', max_hops=3, campaign_filter='IBS-2024-A')
df11_sum


**What to look at.** The 3-hop chain (`SC-MTY -> SC-LAX -> SC-DAL -> SC-SPB`) lights up automatically; you didn't write a join per level. The same query works at length 1, 5, or 17 - Pathfinder enumerates them all in one shot. Plain SQL would need a recursive CTE; PyRel exposes path traversal as a first-class operator over an N-arity edge.


## Act 7 - Multi-reasoner: Louvain communities + MIP load-balance

> **The question:** 'Use the cohorts the graph found to fairly balance the schedule - no cohort should be pushed to weeks 3-4 more than 40% of the time.'

This is the explicit multi-reasoner moment. Louvain (Graph reasoner, from Act 2b) groups vehicles into BOM-sharing cohorts. Those labels propagate onto every recall, and then a fresh MIP (Prescriptive reasoner) minimises urgency-weighted lateness subject to a per-community late-share cap. The cap value comes from the graph; the schedule comes from the MIP; both share the same ontology.


In [ ]:
from rai_code.manual.demo_queries import q12_balanced_schedule

df12, si12, by_comm12 = q12_balanced_schedule(max_late_share=0.40)
print(f'status: {si12.termination_status}    objective: {si12.objective_value:.2f}    '
      f'solve_time: {si12.solve_time_sec:.1f}s    jobs assigned: {len(df12)}')
by_comm12


In [ ]:
# Stacked bar: jobs per (community, week). Visualises that no community
# absorbs more than the late-share cap of week-3+ work.
df12_pivot = df12.groupby(['community', 'week']).size().reset_index(name='jobs')
fig = px.bar(
    df12_pivot, x='community', y='jobs', color='week',
    color_continuous_scale='Viridis',
    title='Q12: jobs scheduled per community x week (Louvain + MIP, max_late_share=40%)',
)
fig.update_layout(xaxis_title='Louvain community', yaxis_title='jobs')
fig.show()


**Why this matters.** Two reasoners cooperating through the same ontology - the Graph reasoner discovers the community structure, the Prescriptive reasoner respects it as a constraint. That composition is exactly what makes RAI a multi-reasoner system: you swap structural insight directly into decision logic without leaving the language.


**Closing.** Seven acts, six reasoners, one ontology, one schema. The rule the operator wrote in Act 5 (`PriorityVehicle = OpenRecall + PriorAccident`) is now structural - it lights up in Act 1 (those VINs breach SLA more visibly), in Act 3 (urgency score already weights `accident_severity`), and in Act 4 (the LP re-solves with the constraint). Acts 6 and 7 show that the same ontology supports path-traversal queries over 3-way relationships and direct composition of Graph + Prescriptive reasoners. The institutional knowledge moved from a senior manager's spreadsheet to the data model itself.
